In [ ]:
#SQL
import sqlite3
import pandas as pd

conn = sqlite3.connect("noleggio_biciclette1.db")
cursor = conn.cursor()


In [ ]:
# Creazione Tabella Clienti
cursor.execute('''
CREATE TABLE Clienti (
    ID_Cliente INT PRIMARY KEY,
    Nome VARCHAR(30) NOT NULL,
    Cognome VARCHAR(30) NOT NULL,
    Telefono VARCHAR(20) UNIQUE,
    Email VARCHAR(50) UNIQUE
);
''')

# Creazione Tabella Punti di gestione
cursor.execute('''
CREATE TABLE Punti_di_gestione(
    ID_Stazione INT PRIMARY KEY,
    Nome_Stazione VARCHAR(40) NOT NULL,
    Via VARCHAR(40) NOT NULL,
    CAP VARCHAR(5) NOT NULL,
    Civico VARCHAR(10) NOT NULL,
    Città VARCHAR(255) NOT NULL
);
''')

# Creazione Tabella Biciclette
cursor.execute('''
CREATE TABLE Biciclette(
    ID_Bici INT PRIMARY KEY,
    Modello VARCHAR(255) NOT NULL,
    Stato_utilizzo VARCHAR(255) NOT NULL,
    Tipologia VARCHAR(255) NOT NULL,
    Tariffa_oraria DECIMAL(6,2) NOT NULL
);
''')

# Creazione Sottoclasse Bici Elettrica
cursor.execute('''
CREATE TABLE Bici_elettrica(
    ID_Bici INT PRIMARY KEY,
    Potenza_motore INT NOT NULL,
    Stato_Carica INT NOT NULL CHECK (Stato_Carica BETWEEN 0 AND 100),
    Tipo_Pneumatici VARCHAR(255) NOT NULL,
    FOREIGN KEY (ID_Bici)
        REFERENCES Biciclette(ID_Bici)
        ON DELETE CASCADE
        ON UPDATE CASCADE
);
''')

# Creazione Sottoclasse Bici Muscolare
cursor.execute('''
CREATE TABLE Bici_muscolare(
    ID_Bici INT PRIMARY KEY,
    Num_Rapporti INT NOT NULL,
    Tipo_Pneumatici VARCHAR(255) NOT NULL,
    FOREIGN KEY (ID_Bici)
        REFERENCES Biciclette(ID_Bici)
        ON DELETE CASCADE
        ON UPDATE CASCADE
);
''')

# Creazione Tabella Noleggi
cursor.execute('''
CREATE TABLE Noleggi(
    ID_Noleggio INT PRIMARY KEY,
    Data_inizio DATE NOT NULL,
    Ora_inizio TIME NOT NULL,
    Data_fine DATE,
    Ora_fine TIME,
    Importo_totale_pagato DECIMAL(6,2),

    Cliente INT NOT NULL,
    Bicicletta INT NOT NULL,
    Stazione_ritiro INT NOT NULL,
    Stazione_riconsegna INT,

    FOREIGN KEY (Cliente)
        REFERENCES Clienti(ID_Cliente)
        ON DELETE NO ACTION
        ON UPDATE CASCADE,

    FOREIGN KEY (Bicicletta)
        REFERENCES Biciclette(ID_Bici)
        ON DELETE NO ACTION
        ON UPDATE CASCADE,

    FOREIGN KEY (Stazione_ritiro)
        REFERENCES Punti_di_gestione(ID_Stazione)
        ON DELETE NO ACTION
        ON UPDATE CASCADE,

    FOREIGN KEY (Stazione_riconsegna)
        REFERENCES Punti_di_gestione(ID_Stazione)
        ON DELETE NO ACTION
        ON UPDATE CASCADE,

    CHECK (Data_fine IS NULL OR Data_fine >= Data_inizio)
);
''')


In [ ]:
# Inserimento Clienti
cursor.executemany("INSERT INTO Clienti VALUES (?, ?, ?, ?, ?)", [
    (1, "Mario", "Rossi", "333123456", "mario.rossi@email.com"),
    (2, "Anna", "Verdi", "333987654", "anna.verdi@email.com")
])

# Inserimento Stazioni
cursor.executemany("INSERT INTO Punti_di_gestione VALUES (?, ?, ?, ?, ?, ?)", [
    (100, "Stazione Centrale", "Via Roma", "70100", "10", "Bari"),
    (101, "Stazione Politecnico", "Via Amendola", "70125", "126", "Bari")
])

# Inserimento Biciclette generiche
cursor.executemany("INSERT INTO Biciclette VALUES (?, ?, ?, ?, ?)", [
    (10, "E-Bike Spark 200", "Disponibile", "Elettrica", 4.50),
    (11, "Rockrider St 100", "In Uso", "Muscolare", 2.00)
])

# Popolamento delle specializzazioni
cursor.execute("INSERT INTO Bici_elettrica VALUES (10, 250, 85, 'Stradali Rinforzati')")
cursor.execute("INSERT INTO Bici_muscolare VALUES (11, 21, 'Tassellati MTB')")

# Inserimento Noleggi
cursor.executemany("INSERT INTO Noleggi VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)", [
    (500, "2026-07-10", "09:00", "2026-07-12", "18:00", 45.00, 1, 10, 100, 101),
    (501, "2026-07-15", "10:30", "2026-07-15", "12:30", 4.00, 1, 11, 100, 100),
    (502, "2026-07-18", "08:00", None, None, None, 2, 11, 101, None)
])

conn.commit()


In [ ]:
#NOLEGGIO PER UN CLIENTE
query_1 = '''
SELECT
    C.Nome,
    C.Cognome,
    B.Modello,
    CAST((julianday(N.Data_fine) - julianday(N.Data_inizio)) AS INT) AS Durata_Giorni
FROM
    Noleggi AS N
    INNER JOIN Clienti AS C ON N.Cliente = C.ID_Cliente
    INNER JOIN Biciclette AS B ON N.Bicicletta = B.ID_Bici
WHERE
    C.ID_Cliente = 1;
'''
df_res1 = pd.read_sql_query(query_1, conn)
print(df_res1.to_string(index=False))
print("\n" + "-"*50 + "\n")


#CONTEGGIO RITIRI PER OGNI STAZIONE
query_2 = '''
SELECT
    P.ID_Stazione,
    P.Nome_Stazione,
    COUNT(N.ID_Noleggio) AS Numero_Ritiri
FROM
    Punti_di_gestione AS P
    INNER JOIN Noleggi AS N ON P.ID_Stazione = N.Stazione_ritiro
GROUP BY
    P.ID_Stazione,
    P.Nome_Stazione;
'''
df_res2 = pd.read_sql_query(query_2, conn)
print(df_res2.to_string(index=False))

conn.close()

 Nome Cognome          Modello  Durata_Giorni
Mario   Rossi E-Bike Spark 200              2
Mario   Rossi Rockrider St 100              0

--------------------------------------------------

 ID_Stazione        Nome_Stazione  Numero_Ritiri
         100    Stazione Centrale              2
         101 Stazione Politecnico              1
